In [ ]:
# 1. Settings. RUNS may be shortened to divide years/runs across runtimes.
import os, sys, json, time, shutil, zipfile, hashlib, subprocess, threading, re
from pathlib import Path
UPLOAD = Path("/content")
WORK = Path("/content/fv_stream_px")
CODE = WORK / "code"
bundle = json.loads((UPLOAD / "bundle.json").read_text())
RUNS = [run["name"] for run in bundle["runs"]]  # e.g. ["at_2022", "barc_2021"]
WORKERS = 32 if (os.cpu_count() or 1) >= 8 else 16
DRIVE_BACKUP = Path("/content/drive/MyDrive/fieldvision_stream_px")
needed = [bundle["code_zip"], *[r["fields"] for r in bundle["runs"]],
          *[r["reference"] for r in bundle["runs"] if r["reference"]]]
missing = [name for name in needed if not (UPLOAD / name).is_file()]
print("missing uploads:", missing or "none")
print(f"{os.cpu_count()} CPUs, {WORKERS} read threads; Python {sys.version.split()[0]}")
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Optional, recommended: mount Google Drive so checkpoints survive a recycled runtime
USE_DRIVE = True  # Set False to run without Drive; download results before the runtime ends.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Install pinned binary wheels and report the actual runtime versions.
pins = [f"{name}=={version}" for name, version in bundle["pins"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", *pins], check=True)
version_code = (
    "import importlib.metadata as m, rasterio, pyproj, sys;"
    "print('python', sys.version.split()[0]);"
    f"[print(p, m.version(p)) for p in {list(bundle['pins'])!r}];"
    "print('GDAL', rasterio.__gdal_version__); print('PROJ', pyproj.proj_version_str)"
)
versions = subprocess.run([sys.executable, "-c", version_code],
                          capture_output=True, text=True, check=True).stdout
print(versions)
print("Pin source:", bundle["pins_source"])

In [ ]:
# 4. Verify the upload ZIP, unpack Python sources, and verify frozen Sentinel-2 code.
def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

code_zip = UPLOAD / bundle["code_zip"]
assert file_hash(code_zip) == bundle["code_sha256"], "Code ZIP checksum differs"
with zipfile.ZipFile(code_zip) as archive:
    for member in archive.infolist():
        parts = member.filename.split("/")
        assert parts[0] == "fieldvision" and member.filename.endswith(".py")
        assert not any(p in ("", ".", "..") or ":" in p or "\\" in p for p in parts)
    shutil.rmtree(CODE, ignore_errors=True)
    archive.extractall(CODE)
digest = hashlib.sha256()
for path in sorted((CODE / "fieldvision" / "s2").glob("*.py")):
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
s2_digest = digest.hexdigest()
assert s2_digest == bundle["s2_source_sha256"], "Frozen Sentinel-2 fingerprint differs"
print("code ZIP", bundle["code_sha256"], "| frozen s2", s2_digest)
env = dict(os.environ, PYTHONPATH=str(CODE), PYTHONUNBUFFERED="1")

In [ ]:
# 5. Extract selected runs sequentially. Rerun this cell to resume unfinished blocks.
selected = {run["name"]: run for run in bundle["runs"]}
assert len(RUNS) == len(set(RUNS)) and set(RUNS) <= set(selected), "Unknown or duplicate RUNS"
CHECK_PASSED = set()
exit_codes = {}

def read_json(path):
    return json.loads(path.read_text())

def copy_atomic(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".tmp")
    shutil.copyfile(source, temporary)
    temporary.replace(target)

def copy_finished(source, target, restoring=False):
    if not source.exists():
        return
    source_manifest = source / "manifest.json"
    target_manifest = target / "manifest.json"
    if source_manifest.exists() and target_manifest.exists():
        if read_json(source_manifest).get("signature") != read_json(target_manifest).get("signature"):
            raise RuntimeError(f"Checkpoint signature mismatch for {target.name}")
    for status_path in sorted((source / "blocks").glob("*.json")):
        status_bytes = status_path.read_bytes()
        if json.loads(status_bytes).get("status") not in ("complete", "partial"):
            continue
        target_status = target / "blocks" / status_path.name
        if restoring and target_status.exists():
            local = read_json(target_status)
            if local.get("status") in ("complete", "partial") and all(
                    target_status.with_suffix(suffix).exists() for suffix in (".parquet", ".npz")):
                continue  # Existing finished local work takes precedence over a Drive snapshot.
        files = [status_path.with_suffix(suffix) for suffix in (".parquet", ".npz")]
        if not all(path.is_file() for path in files):
            continue
        target_status.parent.mkdir(parents=True, exist_ok=True)
        staged = []
        try:
            for path in files:
                dest = target_status.with_suffix(path.suffix)
                temp = dest.with_name(dest.name + ".tmp")
                shutil.copyfile(path, temp)
                staged.append((temp, dest))
            if status_path.read_bytes() != status_bytes:
                continue
            for temp, dest in staged:
                temp.replace(dest)
            temp_status = target_status.with_name(target_status.name + ".tmp")
            temp_status.write_bytes(status_bytes)
            temp_status.replace(target_status)
            catalog = source / "catalog" / status_path.name
            if catalog.exists():
                copy_atomic(catalog, target / "catalog" / catalog.name)
        finally:
            for temp, _ in staged:
                temp.unlink(missing_ok=True)
    if source_manifest.exists() and (not restoring or not target_manifest.exists()):
        copy_atomic(source_manifest, target_manifest)

def progress(out):
    manifest = read_json(out / "manifest.json") if (out / "manifest.json").exists() else {}
    count = sum(read_json(path).get("status") in ("complete", "partial")
                for path in (out / "blocks").glob("*.json"))
    return f"{count}/{manifest.get('block_count', '?')}"

for name in RUNS:
    run = selected[name]
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    if USE_DRIVE:
        copy_finished(DRIVE_BACKUP / name, out, restoring=True)
    stop = threading.Event()
    backup_errors = []
    def backup_loop():
        while not stop.wait(300):
            try:
                copy_finished(out, DRIVE_BACKUP / name)
            except Exception as exc:
                backup_errors.append(str(exc))
                print(f"{name}: Drive backup failed: {exc}", flush=True)
    thread = threading.Thread(target=backup_loop, daemon=True) if USE_DRIVE else None
    if thread:
        thread.start()
    cmd = [sys.executable, "-W", "ignore", "-m", "fieldvision.stream.pixels",
           "--fields", str(UPLOAD / run["fields"]), "--year", str(run["year"]),
           "--out", str(out), "--chip", str(bundle["chip_size"]), "--workers", str(WORKERS)]
    print(f"[{time.strftime('%H:%M:%S')}] {name} {progress(out)} blocks done | starting", flush=True)
    proc = None
    try:
        with (out / "run.log").open("a") as log:
            proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                log.write(line)
                if any(word in line for word in ("Block ", "complete:", "partial:", "ERROR", "WARNING")):
                    print(f"[{time.strftime('%H:%M:%S')}] {name} {progress(out)} blocks done | {line.strip()}", flush=True)
            exit_codes[name] = proc.wait()
    except BaseException:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        raise
    finally:
        stop.set()
        if thread:
            thread.join()
            try:
                copy_finished(out, DRIVE_BACKUP / name)
            except Exception as exc:
                print(f"{name}: final Drive backup failed: {exc}")
    manifest = read_json(out / "manifest.json") if (out / "manifest.json").exists() else {}
    errors = {}
    for checkpoint in (out / "blocks").glob("*.json"):
        for error in read_json(checkpoint).get("scene_errors", {}).values():
            network = any(token in error for token in ("RasterioIOError", "CPLE_", "Timeout", "Connection", "HTTP"))
            kind = "network read (rerun this cell)" if network else error.split(":", 1)[0]
            errors[kind] = errors.get(kind, 0) + 1
    print(f"[{time.strftime('%H:%M:%S')}] {name} {progress(out)} blocks done | exit {exit_codes[name]}")
    print(name, "scene errors:", errors or "none")
print("Exit codes (0 complete, 2 partial):", exit_codes)
assert all(code in (0, 2) for code in exit_codes.values()), "Extraction failed; inspect the log and rerun this cell"

In [ ]:
# 6. Check European field/date medians against the supplied references; print BARC counts only.
for name in RUNS:
    run = selected[name]
    out = WORK / name
    manifest = read_json(out / "manifest.json")
    counts = {key: manifest.get(key) for key in ("status", "input_fields", "fields_with_chips",
              "n_obs", "median_chips_per_field", "scene_failures")}
    print(name, json.dumps(counts))
    if run["reference"]:
        result = subprocess.run([sys.executable, "-m", "fieldvision.stream.pixels", "check",
                                 "--run", str(out), "--reference", str(UPLOAD / run["reference"]),
                                 "--min-agree", ".99"], env=env, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode:
            print(result.stderr)
            raise RuntimeError(f"{name}: same-result check FAILED; stop before exporting")
        CHECK_PASSED.add(name)
        print(name, "same-result check PASSED")

In [ ]:
# 7. Export per-run ZIPs, <=50 MiB parts and SHA256SUMS; copy to Drive and print sizes.
for name in RUNS:
    run = selected[name]
    assert not run["reference"] or name in CHECK_PASSED, f"Run step 6 successfully for {name}"
    out = WORK / name
    result = UPLOAD / f"stream_px_{name}_colab_chips.zip"
    environment = versions + f"\ncode ZIP {bundle['code_sha256']}\nfrozen s2 {s2_digest}\n"
    environment += "same-result check PASSED\n" if name in CHECK_PASSED else "same-result check not applicable: no reference\n"
    with zipfile.ZipFile(result, "w", zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for filename in ("chips.npz", "timeseries.parquet", "manifest.json"):
            archive.write(out / filename, filename)
        archive.writestr("colab_environment.txt", environment)
    for old in UPLOAD.glob(result.name + ".*"):
        if re.fullmatch(re.escape(result.name) + r"\.\d{3,}", old.name):
            old.unlink()
    parts, lines = [], []
    whole = hashlib.sha256()
    with result.open("rb") as source:
        index = 0
        while True:
            first = source.read(1024 * 1024)
            if not first:
                break
            part = result.with_name(result.name + f".{index:03d}")
            digest = hashlib.sha256()
            remaining = bundle["result_part_mb"] * 1024 * 1024
            with part.open("wb") as target:
                chunk = first
                while chunk:
                    target.write(chunk)
                    digest.update(chunk)
                    whole.update(chunk)
                    remaining -= len(chunk)
                    if not remaining:
                        break
                    chunk = source.read(min(1024 * 1024, remaining))
            parts.append(part)
            lines.append(f"{digest.hexdigest()}  {part.name}\n")
            index += 1
    lines.append(f"{whole.hexdigest()}  {result.name}\n")
    sums = UPLOAD / f"stream_px_{name}_SHA256SUMS.txt"
    sums.write_text("".join(lines), encoding="utf-8")
    exports = [result, *parts, sums]
    if USE_DRIVE:
        current_parts = {part.name for part in parts}
        for old in DRIVE_BACKUP.glob(result.name + ".*"):
            if re.fullmatch(re.escape(result.name) + r"\.\d{3,}", old.name) and old.name not in current_parts:
                old.unlink()
        for path in exports:
            copy_atomic(path, DRIVE_BACKUP / path.name)
    for path in exports:
        print(f"{path.name} {path.stat().st_size:,} bytes" + (" | copied to Drive" if USE_DRIVE else ""))
print("Download every .zip.NNN part and its SHA256SUMS.txt; the full ZIP is optional.")